In [1]:
# %pip install bertopic huggingface_hub scikit-learn
# %pip install --upgrade llama-cpp-python

import pandas as pd
from bertopic import BERTopic
from bertopic.representation import LlamaCPP
from llama_cpp import Llama
from huggingface_hub import hf_hub_download

In [2]:
PARQUET_PATH = r"C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\ADS_Pipeline\runs\run_20260310_180458_ADS_Curation_Run\data\publications_disambiguated.parquet"

df = pd.read_parquet(PARQUET_PATH)
docs = df["full_text"].dropna().astype(str).str.strip()
docs = docs[docs != ""].tolist()

# Optional: Für den MWE-Test auf 500 Dokumente begrenzt. Ggf. anpassen.
docs = docs[:500]
print(f"Loaded {len(docs)} documents")

Loaded 382 documents


In [3]:
PHYSICS_PROMPT = """\
You are an experienced researcher in gravitational physics, astrophysics, \
and cosmology. You are labeling research topic clusters based on scientific \
abstracts.

Documents: [DOCUMENTS]
Keywords: [KEYWORDS]

Task:
- Generate EXACTLY ONE topic label.
- The label MUST contain between 4 and 7 words (inclusive).
- The label should read like a review article or textbook chapter title.

Guidelines:
- Use standard scientific terminology from the field.
- Be specific about the physical phenomenon or method.
- Avoid generic terms like "studies", "analysis", or "research".

Output format (single line):
topic: <label>

Do NOT write anything else (no explanations, no additional sentences, \
no quotes, no bullet points)."""

model_path = hf_hub_download(
    repo_id="Qwen/Qwen2.5-0.5B-Instruct-GGUF",
    filename="qwen2.5-0.5b-instruct-q4_k_m.gguf",
)

llm = Llama(
    model_path=model_path,
    n_gpu_layers=0,  # 0 = CPU only
    n_ctx=4096,
    verbose=False,
)

representation_model = LlamaCPP(llm, prompt=PHYSICS_PROMPT)

llama_context: n_ctx_per_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


In [4]:
topic_model = BERTopic(
    representation_model=representation_model,
    verbose=True
)

topics, probs = topic_model.fit_transform(docs)

topic_model.get_topic_info()

2026-03-11 15:02:02,645 - BERTopic - Embedding - Transforming documents to embeddings.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

2026-03-11 15:02:11,458 - BERTopic - Embedding - Completed ✓
2026-03-11 15:02:11,462 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-11 15:02:23,857 - BERTopic - Dimensionality - Completed ✓
2026-03-11 15:02:23,859 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-11 15:02:23,879 - BERTopic - Cluster - Completed ✓
2026-03-11 15:02:23,886 - BERTopic - Representation - Fine-tuning topics using representation models.
100%|██████████| 3/3 [00:55<00:00, 18.50s/it]
2026-03-11 15:03:19,470 - BERTopic - Representation - Completed ✓


,Topic,Count,Name,Representation,Representative_Docs
0,-1,15,-1_topic: Eddington's Large Cosmic Numbers and...,[topic: Eddington's Large Cosmic Numbers and Q...,[On the Correlations between the Particles in ...
1,0,343,0_topic: The Quantum Gravity Thesis: A Review ...,[topic: The Quantum Gravity Thesis: A Review o...,[On general-relativistic and gauge field theor...
2,1,24,"1_topic: Helmholtz Vorticity Theorem, General-...","[topic: Helmholtz Vorticity Theorem, General-R...",[On the General-Relativistic and Covariant For...
